### Defining "Like" vs. "Dislike"
Since your data doesn't have a "Thumbs Down" button, we can use proxy logic:
- Like (1): If you listened to more than 30 seconds (or a high percentage of the song).
- Dislike (0): If you skipped it within the first few seconds (e.g., msPlayed < 30000).

#### 1. The Strategy
To predict new songs you'll like, we need to build a Binary Classifier.
- Labeling: Create a target column based on msPlayed.
- Feature Enrichment: Use your Client ID/Secret to get the "DNA" of your favorite tracks (Audio Features).
- The Model: We'll use Random Forest because it can tell us why it made a choice (e.g., "You like songs with high energy but low acousticness").

##### Part A: Consolidate and Label

In [1]:
# import necessary libraries
import pandas as pd
import glob
import json

In [2]:
# load all JSON files
files = glob.glob("spotify_data/*.json")
data_list = []

for file in files:
    with open(file, 'r', encoding='utf-8') as f:
        data_list.extend(json.load(f))

df = pd.DataFrame(data_list)

# create the Target variable
df['target'] = (df['msPlayed'] > 30000).astype(int)

print(f"Dataset size: {len(df)}")
print(df['target'].value_counts()) # see how many likes vs skips you have

Dataset size: 30812
target
0    15816
1    14996
Name: count, dtype: int64


##### Part B: Enrich with Spotify Audio Features

In [10]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import time
import os
from dotenv import load_dotenv
import requests

In [11]:
# load your credentials from the .env file
load_dotenv()
client_id = os.getenv("SPOTIPY_CLIENT_ID") 
client_secret = os.getenv("SPOTIPY_CLIENT_SECRET") 

session = requests.Session()
session.proxies = {}

auth_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
sp = spotipy.Spotify(auth_manager=auth_manager, requests_session=session)

try:
    test_search = sp.search(q='track:Espresso artist:Sabrina Carpenter', type='track', limit=1)
    print(f"Success! Found: {test_search['tracks']['items'][0]['name']}")
except Exception as e:
    print(f"Still failing. Error: {e}")

Success! Found: Espresso


In [12]:
def fetch_features_safe(row):
    # Skip if data is missing or "Unknown"
    if pd.isna(row['trackName']) or "Unknown" in str(row['trackName']):
        return None
        
    try:
        query = f"track:{row['trackName']} artist:{row['artistName']}"
        results = sp.search(q=query, type='track', limit=1)
        
        if results['tracks']['items']:
            track_id = results['tracks']['items'][0]['id']
            # Important: this is a second API call
            features = sp.audio_features(track_id)[0]
            return features
        return None
    except:
        return None

#### 2. Data cleaning and preprocessing
Before the model sees the data, we must:
- Remove IDs: The model shouldn't learn the track_id or uri.
- Scale Data: Features like loudness (usually -60 to 0) and tempo (60 to 200) have different scales. We should normalize them.

In [15]:
import time
import pandas as pd

# 1. Get unique tracks
unique_tracks = df[['trackName', 'artistName']].drop_duplicates().head(100).copy()
print(f"Fetching features for {len(unique_tracks)} tracks...")

all_features = []
found_indices = []

# 2. Iterate manually to catch empty results
for index, row in unique_tracks.iterrows():
    feat = fetch_features_safe(row)
    if feat is not None:
        all_features.append(feat)
        found_indices.append(index)
    # Adding a small sleep to prevent rate limiting
    time.sleep(0.1)

# 3. Create the features DataFrame
if len(all_features) > 0:
    features_df = pd.DataFrame(all_features)
    
    # Filter the original unique_tracks to match only what we found
    valid_names = unique_tracks.loc[found_indices].reset_index(drop=True)
    
    # 4. Combine
    full_enriched = pd.concat([valid_names, features_df], axis=1)
    
    print("Success! Columns found:", full_enriched.columns.tolist())
    print(full_enriched[['trackName', 'artistName', 'danceability', 'energy']].head())
else:
    print("Zero features were returned. Check if 'fetch_features_safe' is returning 'None'.")

# 4. Expand the features dictionary into columns
features_list = list(valid_tracks['features'])
features_df = pd.DataFrame(features_list)

# 5. Combine names with features
# We drop 'features' (the dict column) and glue the new columns on
full_enriched = pd.concat([
    valid_tracks.drop(columns=['features']).reset_index(drop=True), 
    features_df.reset_index(drop=True)
], axis=1)

print("Data enriched! Preview:")
print(full_enriched[['trackName', 'artistName', 'danceability', 'energy']].head())

Fetching features for 100 tracks...


HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=7JgNAnCjJvL8hBR1kmCOFF with Params: {} returned 403 due to None
HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=0FTpFi1BlqoBVELlh7jK50 with Params: {} returned 403 due to None
HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=6M0EP76qW9TG4SNU0agu94 with Params: {} returned 403 due to None
HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=4B1tz27AMeVxHzcPkZSCLY with Params: {} returned 403 due to None
HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=6D8y7Bck8h11byRY88Pt2z with Params: {} returned 403 due to None
HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=059qzhNaJb8tsqpdl2bHfF with Params: {} returned 403 due to None
HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=4bDIAWaOBGAAk95nyXI6zJ with Params: {} returned 403 due to None
HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=

Zero features were returned. Check if 'fetch_features_safe' is returning 'None'.
Data enriched! Preview:


KeyError: "['danceability', 'energy'] not in index"

In [ ]:
# Merge the features back to the main history using track and artist names
final_df = pd.merge(df, full_enriched, on=['trackName', 'artistName'])

# List of columns the model will use to "learn" your taste
feature_cols = [
    'danceability', 'energy', 'key', 'loudness', 'mode', 
    'speechiness', 'acousticness', 'instrumentalness', 
    'liveness', 'valence', 'tempo'
]

# X = The technical features, y = Did you like it (target)?
X = final_df[feature_cols]
y = final_df['target']

print(f"Final training set size: {X.shape}")

#### 3. Training the model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 1. Performance Report
y_pred = model.predict(X_test)
print("--- Model Performance ---")
print(classification_report(y_test, y_pred))

# 2. Taste Analysis: What matters most to you?
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\n--- Your Personal Taste Drivers ---")
print(importances)

KeyError: "None of [Index(['danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',\n       'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'],\n      dtype='object')] are in the [columns]"